In [1]:
import pandas as pd

from geo_utils import (
    create_bigquery_client,
    create_nominatim_geocoder,
    geocode_location,
    load_team_locations,
)

In [2]:
OUTPUT_TABLE = "PracticeFacility"

SOURCE = "Manual reference + OpenStreetMap Nominatim"

In [3]:
PRACTICE_FACILITIES = {
    "Anaheim Ducks": {
        "facility_name": "Great Park Ice & FivePoint Arena",
        "facility_type": "Dedicated practice facility",
        "address": "888 Ridge Valley",
        "city": "Irvine",
        "state_province": "California",
        "country": "United States",
        "notes": "Official Ducks practice facility.",
    },
    "Boston Bruins": {
        "facility_name": "Warrior Ice Arena",
        "facility_type": "Dedicated practice facility",
        "address": "90 Guest Street",
        "city": "Boston",
        "state_province": "Massachusetts",
        "country": "United States",
        "notes": "Official Bruins practice facility.",
    },
    "Buffalo Sabres": {
        "facility_name": "LECOM Harborcenter",
        "facility_type": "Arena-connected",
        "address": "100 Washington Street",
        "city": "Buffalo",
        "state_province": "New York",
        "country": "United States",
        "notes": "Connected to KeyBank Center.",
    },
    "Calgary Flames": {
        "facility_name": "Scotiabank Saddledome",
        "facility_type": "Arena",
        "address": "555 Saddledome Rise SE",
        "city": "Calgary",
        "state_province": "Alberta",
        "country": "Canada",
        "notes": "Current practice location until Scotia Place opens.",
    },
    "Carolina Hurricanes": {
        "facility_name": "Wake Competition Center",
        "facility_type": "Dedicated practice facility",
        "address": "801 Corporate Center Drive",
        "city": "Morrisville",
        "state_province": "North Carolina",
        "country": "United States",
        "notes": "Formerly known as Invisalign Arena.",
    },
    "Chicago Blackhawks": {
        "facility_name": "Fifth Third Arena",
        "facility_type": "Dedicated practice facility",
        "address": "1801 West Jackson Boulevard",
        "city": "Chicago",
        "state_province": "Illinois",
        "country": "United States",
        "notes": None,
    },
    "Colorado Avalanche": {
        "facility_name": "Family Sports Center",
        "facility_type": "Dedicated practice facility",
        "address": "6901 South Peoria Street",
        "city": "Centennial",
        "state_province": "Colorado",
        "country": "United States",
        "notes": None,
    },
    "Columbus Blue Jackets": {
        "facility_name": "OhioHealth Ice Haus at Nationwide Arena",
        "facility_type": "Arena-connected",
        "address": "200 West Nationwide Boulevard",
        "city": "Columbus",
        "state_province": "Ohio",
        "country": "United States",
        "notes": "Connected to Nationwide Arena.",
    },
    "Dallas Stars": {
        "facility_name": "Comerica Center",
        "facility_type": "Dedicated practice facility",
        "address": "2601 Avenue of the Stars",
        "city": "Frisco",
        "state_province": "Texas",
        "country": "United States",
        "notes": None,
    },
    "Detroit Red Wings": {
        "facility_name": "Little Caesars Arena",
        "facility_type": "Arena-connected",
        "address": "2645 Woodward Avenue",
        "city": "Detroit",
        "state_province": "Michigan",
        "country": "United States",
        "notes": "BELFOR Training Center is inside the arena.",
    },
    "Edmonton Oilers": {
        "facility_name": "Downtown Community Arena",
        "facility_type": "Arena-connected",
        "address": "10220 104 Avenue NW",
        "city": "Edmonton",
        "state_province": "Alberta",
        "country": "Canada",
        "notes": "Connected to Rogers Place.",
    },
    "Florida Panthers": {
        "facility_name": "Baptist Health IcePlex",
        "facility_type": "Dedicated practice facility",
        "address": "800 Northeast 8th Street",
        "city": "Fort Lauderdale",
        "state_province": "Florida",
        "country": "United States",
        "notes": None,
    },
    "Los Angeles Kings": {
        "facility_name": "Toyota Sports Performance Center",
        "facility_type": "Dedicated practice facility",
        "address": "555 North Nash Street",
        "city": "El Segundo",
        "state_province": "California",
        "country": "United States",
        "notes": None,
    },
    "Minnesota Wild": {
        "facility_name": "TRIA Rink at Treasure Island Center",
        "facility_type": "Dedicated practice facility",
        "address": "400 Wabasha Street North",
        "city": "Saint Paul",
        "state_province": "Minnesota",
        "country": "United States",
        "notes": None,
    },
    "Montréal Canadiens": {
        "facility_name": "CN Sports Complex",
        "facility_type": "Dedicated practice facility",
        "address": "8000 Boulevard Leduc",
        "city": "Brossard",
        "state_province": "Quebec",
        "country": "Canada",
        "notes": None,
    },
    "Nashville Predators": {
        "facility_name": "Ford Ice Center Bellevue",
        "facility_type": "Dedicated practice facility",
        "address": "7638 Highway 70 South",
        "city": "Nashville",
        "state_province": "Tennessee",
        "country": "United States",
        "notes": "Primary practice facility.",
    },
    "New Jersey Devils": {
        "facility_name": "Prudential Center",
        "facility_type": "Arena-connected",
        "address": "25 Lafayette Street",
        "city": "Newark",
        "state_province": "New Jersey",
        "country": "United States",
        "notes": "RWJBarnabas Health Hockey House is attached to Prudential Center.",
    },
    "New York Islanders": {
        "facility_name": "Northwell Health Ice Center",
        "facility_type": "Dedicated practice facility",
        "address": "200 Merrick Avenue",
        "city": "East Meadow",
        "state_province": "New York",
        "country": "United States",
        "notes": None,
    },
    "New York Rangers": {
        "facility_name": "MSG Training Center",
        "facility_type": "Dedicated practice facility",
        "address": "600 Corporate Court",
        "city": "Greenburgh",
        "state_province": "New York",
        "country": "United States",
        "notes": None,
    },
    "Ottawa Senators": {
        "facility_name": "Bell Sensplex",
        "facility_type": "Dedicated practice facility",
        "address": "1565 Maple Grove Road",
        "city": "Ottawa",
        "state_province": "Ontario",
        "country": "Canada",
        "notes": None,
    },
    "Philadelphia Flyers": {
        "facility_name": "Flyers Training Center",
        "facility_type": "Dedicated practice facility",
        "address": "601 Laurel Oak Road",
        "city": "Voorhees",
        "state_province": "New Jersey",
        "country": "United States",
        "notes": None,
    },
    "Pittsburgh Penguins": {
        "facility_name": "UPMC Lemieux Sports Complex",
        "facility_type": "Dedicated practice facility",
        "address": "8000 Cranberry Springs Drive",
        "city": "Cranberry Township",
        "state_province": "Pennsylvania",
        "country": "United States",
        "notes": None,
    },
    "San Jose Sharks": {
        "facility_name": "Sharks Ice at San Jose",
        "facility_type": "Dedicated practice facility",
        "address": "1500 South 10th Street",
        "city": "San Jose",
        "state_province": "California",
        "country": "United States",
        "notes": "Adjacent to Tech CU Arena.",
    },
    "Seattle Kraken": {
        "facility_name": "Kraken Community Iceplex",
        "facility_type": "Dedicated practice facility",
        "address": "10601 5th Avenue NE",
        "city": "Seattle",
        "state_province": "Washington",
        "country": "United States",
        "notes": None,
    },
    "St. Louis Blues": {
        "facility_name": "Centene Community Ice Center",
        "facility_type": "Dedicated practice facility",
        "address": "750 Casino Center Drive",
        "city": "Maryland Heights",
        "state_province": "Missouri",
        "country": "United States",
        "notes": None,
    },
    "Tampa Bay Lightning": {
        "facility_name": "TGH Ice Plex",
        "facility_type": "Dedicated practice facility",
        "address": "10222 Elizabeth Place",
        "city": "Brandon",
        "state_province": "Florida",
        "country": "United States",
        "notes": None,
    },
    "Toronto Maple Leafs": {
        "facility_name": "Ford Performance Centre",
        "facility_type": "Dedicated practice facility",
        "address": "400 Kipling Avenue",
        "city": "Toronto",
        "state_province": "Ontario",
        "country": "Canada",
        "notes": None,
    },
    "Utah Mammoth": {
        "facility_name": "Maverik Center",
        "facility_type": "Temporary practice facility",
        "address": "3200 South Decker Lake Drive",
        "city": "West Valley City",
        "state_province": "Utah",
        "country": "United States",
        "notes": "Temporary training location while the permanent practice facility is developed.",
    },
    "Vancouver Canucks": {
        "facility_name": "Doug Mitchell Thunderbird Sports Centre",
        "facility_type": "Shared practice facility",
        "address": "6066 Thunderbird Boulevard",
        "city": "Vancouver",
        "state_province": "British Columbia",
        "country": "Canada",
        "notes": "Primary shared practice venue.",
    },
    "Vegas Golden Knights": {
        "facility_name": "City National Arena",
        "facility_type": "Dedicated practice facility",
        "address": "1550 South Pavilion Center Drive",
        "city": "Las Vegas",
        "state_province": "Nevada",
        "country": "United States",
        "notes": None,
    },
    "Washington Capitals": {
        "facility_name": "MedStar Capitals Iceplex",
        "facility_type": "Dedicated practice facility",
        "address": "627 North Glebe Road",
        "city": "Arlington",
        "state_province": "Virginia",
        "country": "United States",
        "notes": None,
    },
    "Winnipeg Jets": {
        "facility_name": "Hockey for All Centre",
        "facility_type": "Dedicated practice facility",
        "address": "3969 Portage Avenue",
        "city": "Winnipeg",
        "state_province": "Manitoba",
        "country": "Canada",
        "notes": None,
    },
}

In [4]:
bq = create_bigquery_client()

teams = load_team_locations(
    client=bq,
)

#teams

32 teams loaded from pacey32-agency.Team.OrganizationDetail


In [5]:
missing_reference_teams = sorted(
    set(teams["fullName"])
    - set(PRACTICE_FACILITIES)
)

extra_reference_teams = sorted(
    set(PRACTICE_FACILITIES)
    - set(teams["fullName"])
)

print("Missing:", missing_reference_teams)
print("Extra:", extra_reference_teams)

Missing: []
Extra: []


In [6]:
geocode = create_nominatim_geocoder()

In [7]:
team = "Carolina Hurricanes"

facility = PRACTICE_FACILITIES[team]

facility

{'facility_name': 'Wake Competition Center',
 'facility_type': 'Dedicated practice facility',
 'address': '801 Corporate Center Drive',
 'city': 'Morrisville',
 'state_province': 'North Carolina',
 'country': 'United States',
 'notes': 'Formerly known as Invisalign Arena.'}

In [8]:
result = geocode_location(
    geocode=geocode,
    name=facility["facility_name"],
    address=facility["address"],
    city=facility["city"],
    state_province=facility["state_province"],
    country=facility["country"],
)

result

{'query': 'Wake Competition Center, Morrisville, North Carolina, United States',
 'latitude': 35.8427197,
 'longitude': -78.8176446,
 'matched_address': 'Wake Competition Center, Competition Center Drive, Morrisville, Wake County, North Carolina, 27650, United States',
 'geography_wkt': 'POINT(-78.8176446 35.8427197)',
 'geocode_status': 'FOUND'}

In [14]:
print(geocode("Wake Competition Center"))

Wake Competition Center, McCrimmon Parkway, Morrisville, Wake County, North Carolina, 27650, United States


In [ ]:
print(result["query"])

In [ ]:
rows = []

for _, team in teams.iterrows():

    facility = PRACTICE_FACILITIES[
        team.fullName
    ]

    result = geocode_location(
        geocode=geocode,
        name=facility["facility_name"],
        address=facility["address"],
        city=facility["city"],
        state_province=facility["state_province"],
        country=facility["country"],
    )

    rows.append(
        {
            "team": team.fullName,
            "facility": facility["facility_name"],
            "query": result["query"],
            "matched_address": result["matched_address"],
            "latitude": result["latitude"],
            "longitude": result["longitude"],
            "status": result["geocode_status"],
        }
    )

practice_df = pd.DataFrame(rows)

practice_df

In [ ]:
practice_df[
    practice_df.status != "FOUND"
].sort_values(
    "team"
)

In [ ]:
practice_df[
    practice_df.status == "FOUND"
].sort_values(
    "team"
)